# PROCESAMIENTO

# Procesamiento de Datos para Predicción de Default en Plataforma P2P

## Resumen Ejecutivo

Se procesaron **77,394 registros** de la plataforma Bondora (European P2P Lending), reduciendo de 112 columnas iniciales a **54 columnas** incluyendo la variable objetivo. Se aplicaron técnicas de limpieza, eliminación de leakage temporal, imputación de valores nulos y tratamiento de outliers. La tasa de default final es del **55.3%**.

---

## 1. Planteamiento del Problema

### 1.1 Contexto del Negocio

Las plataformas de préstamos Peer-to-Peer (P2P) como Bondora conectan inversores con prestatarios, eliminando intermediarios financieros tradicionales. Sin embargo, el riesgo de default (incumplimiento de pago) representa la principal amenaza para la rentabilidad de los inversores y la sostenibilidad de la plataforma.

**El desafío:** Evaluar la solvencia de los solicitantes y predecir la probabilidad de default antes de aprobar un préstamo, utilizando datos históricos de la plataforma.

### 1.2 Dataset Utilizado

| Característica | Descripción |
|----------------|-------------|
| **Nombre** | Bondora Public Dataset |
| **Fuente** | https://www.bondora.com/ (datos públicos) |
| **Período** | Marzo 2009 - Enero 2020 (más de 10 años históricos) |
| **Registros iniciales** | ~134,529 filas |
| **Features iniciales** | 112 columnas |
| **Tamaño** | < 100 MB |

### 1.3 Tipo de Problema de Machine Learning

- **Tipo:** Aprendizaje Supervisado
- **Subproblema:** Clasificación Binaria
- **Variable objetivo:** `DEFAULT` (1 = incumple, 0 = paga correctamente)

### 1.4 Creación de la Variable Objetivo

```python
df['DEFAULT'] = df['DefaultDate'].notna().astype(int)

## 2. Configuración Inicial

### 2.1 Importación de Librerias

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configurar visualizaciones
plt.style.use('ggplot')
sns.set_style('whitegrid')
pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 100)

print("Librerías importadas")

Librerías importadas


### 2.2 Carga de Data

In [2]:
import pandas as pd

# Cargar datos (con la ruta correcta desde notebooks/)
df = pd.read_csv('../data/raw/Bondora_raw.csv', low_memory=False)

print(f"Dataset cargado: {df.shape[0]} filas, {df.shape[1]} columnas")
print(f"Memoria aproximada: {df.memory_usage(deep=True).sum() / (1024*1024):.2f} MB")

Dataset cargado: 134529 filas, 112 columnas
Memoria aproximada: 320.06 MB


## 3. Exploración Inicial

#### 3.1 Eliminación de Columnas de Identificación
Se eliminaron columnas que no aportan valor predictivo (IDs, fechas de reporte, etc.):

In [3]:
# Columnas que no sirven para predecir (IDs, fechas de reporte, etc.)
id_columns = [
    'LoanId', 'LoanNumber', 'UserName', 'ReportAsOfEOD',
    'ListedOnUTC', 'BiddingStartedOn', 'LoanApplicationStartedDate',
    'LoanDate', 'ContractEndDate', 'FirstPaymentDate',
    'MaturityDate_Original', 'MaturityDate_Last', 'LastPaymentOn',
    'DebtOccuredOn', 'DebtOccuredOnForSecondary', 'StageActiveSince',
    'GracePeriodStart', 'GracePeriodEnd', 'NextPaymentDate', 'ReScheduledOn'
]

# Eliminar solo las que existen
cols_to_drop = [col for col in id_columns if col in df.columns]
df = df.drop(columns=cols_to_drop)

print(f"Eliminadas {len(cols_to_drop)} columnas de identificación")
print(f"Columnas restantes: {df.shape[1]}")

Eliminadas 20 columnas de identificación
Columnas restantes: 92


#### 3.2 Creación de Variable Objetivo DEFAULT

In [4]:
# Crear DEFAULT correctamente
df['DEFAULT'] = df['DefaultDate'].notna().astype(int)

print("Distribución de DEFAULT:")
print(df['DEFAULT'].value_counts())
print(f"Tasa de default: {df['DEFAULT'].mean()*100:.1f}%")

print("\nValidación: DEFAULT vs Status")
print(pd.crosstab(df['Status'], df['DEFAULT']))

Distribución de DEFAULT:
DEFAULT
0    91614
1    42915
Name: count, dtype: int64
Tasa de default: 31.9%

Validación: DEFAULT vs Status
DEFAULT      0      1
Status               
Current  57014    121
Late      8661  37111
Repaid   25939   5683


#### 3.3 Eliminación de Préstamos Activos

In [5]:
# Eliminar préstamos activos (no sabemos su resultado)
df = df[df['Status'] != 'Current']
print(f"Shape después de eliminar Current: {df.shape}")
print(df['DEFAULT'].value_counts())
print(f"Tasa de default: {df['DEFAULT'].mean()*100:.1f}%")

print("\nValidación: DEFAULT vs Status")
print(pd.crosstab(df['Status'], df['DEFAULT']))

Shape después de eliminar Current: (77394, 93)
DEFAULT
1    42794
0    34600
Name: count, dtype: int64
Tasa de default: 55.3%

Validación: DEFAULT vs Status
DEFAULT      0      1
Status               
Late      8661  37111
Repaid   25939   5683


## 4. Limpieza preliminar de Features

### 4.1 Eliminación de Columnas Constantes
Las columnas con un solo valor no aportan información para el modelo predictivo.

In [6]:
# Eliminar Filas con un solo valor constante

constant_cols = []
for col in df.columns:
    if df[col].nunique() == 1:
        constant_cols.append(col)

df = df.drop(columns=constant_cols)
print(f"Eliminadas {len(constant_cols)} columnas constantes")

Eliminadas 0 columnas constantes


### 4.2 Eliminación de Columnas que tienen nulo

In [7]:
# Eliminar el 10% de Observaciones que tiene nulo

threshold = 0.10
high_null_cols = []

for col in df.columns:
    if col != 'DEFAULT':
        null_pct = df[col].isnull().sum() / len(df)
        if null_pct > threshold:
            high_null_cols.append(col)
            print(f"  {col}: {null_pct*100:.1f}% nulos → eliminar")

df = df.drop(columns=high_null_cols)
print(f"\nEliminadas {len(high_null_cols)} columnas con >10% nulos")

  County: 26.5% nulos → eliminar
  NrOfDependants: 58.2% nulos → eliminar
  EmploymentPosition: 57.7% nulos → eliminar
  WorkExperience: 57.2% nulos → eliminar
  PlannedPrincipalTillDate: 39.8% nulos → eliminar
  CurrentDebtDaysPrimary: 37.3% nulos → eliminar
  CurrentDebtDaysSecondary: 35.7% nulos → eliminar
  DefaultDate: 44.7% nulos → eliminar
  PrincipalOverdueBySchedule: 19.2% nulos → eliminar
  PlannedPrincipalPostDefault: 44.7% nulos → eliminar
  PlannedInterestPostDefault: 44.7% nulos → eliminar
  EAD1: 44.7% nulos → eliminar
  EAD2: 44.7% nulos → eliminar
  PrincipalRecovery: 44.7% nulos → eliminar
  InterestRecovery: 44.7% nulos → eliminar
  RecoveryStage: 34.7% nulos → eliminar
  EL_V0: 94.2% nulos → eliminar
  Rating_V0: 94.2% nulos → eliminar
  EL_V1: 84.4% nulos → eliminar
  Rating_V1: 84.4% nulos → eliminar
  Rating_V2: 69.9% nulos → eliminar
  ActiveLateCategory: 37.3% nulos → eliminar
  WorseLateCategory: 19.8% nulos → eliminar
  CreditScoreEsMicroL: 33.8% nulos → elim

In [8]:
# Analizar Valores con nulos

null_counts = df.isnull().sum()
null_counts = null_counts[null_counts > 0].sort_values(ascending=False)

print(f"Columnas con nulos restantes: {len(null_counts)}")
# print(null_counts.head(50))

Columnas con nulos restantes: 18


### 4.3 Eliminación de Columnas Post-Default (Leakage)
Estas columnas solo existen después de que ocurre el default, causando leakage temporal (usar información del futuro para predecir el pasado).

In [9]:
# Columnas que solo existen después del default
post_default_cols = [
    'CreditScoreFiAsiakasTietoRiskGrade',
    'DefaultDate',
    'PrincipalOverdueBySchedule',
    'PlannedPrincipalPostDefault',
    'PlannedInterestPostDefault',
    'EAD1', 'EAD2',
    'PrincipalRecovery',
    'InterestRecovery',
    'RecoveryStage',
    'ExpectedLoss',
    'LossGivenDefault',
    'ExpectedReturn',
    'ProbabilityOfDefault',
    'PrincipalWriteOffs',
    'InterestAndPenaltyWriteOffs',
    'PrincipalDebtServicingCost',
    'InterestAndPenaltyDebtServicingCost',
    'ActiveLateCategory',
    'ActiveLateLastPaymentCategory',
    'WorseLateCategory',
    'CurrentDebtDaysPrimary',
    'CurrentDebtDaysSecondary',
    'NrOfScheduledPayments',
    'NextPaymentNr'
]

# Eliminar las que existen
cols_to_remove = [col for col in post_default_cols if col in df.columns]
df = df.drop(columns=cols_to_remove)
print(f"Eliminadas {len(cols_to_remove)} columnas post-default")

Eliminadas 4 columnas post-default


## 5. Análisis y Tratamiento de Nulos

### 5.1 Análisis de Nulos Restantes

In [10]:
# Analizar Valores con nulos

null_counts = df.isnull().sum()
null_counts = null_counts[null_counts > 0].sort_values(ascending=False)

print(f"Columnas con nulos restantes: {len(null_counts)}")
print(null_counts.head(50))

Columnas con nulos restantes: 14
MonthlyPayment                       6627
City                                 5044
Rating                               2728
ModelVersion                         2634
HomeOwnershipType                    1652
EmploymentDurationCurrentEmployer     874
EmploymentStatus                      197
OccupationArea                         86
VerificationType                       45
Gender                                 45
MaritalStatus                          45
Education                              45
FreeCash                               45
DebtToIncome                           45
dtype: int64


### 5.2 Imputación de Valores Nulos
Estrategia de imputación:

Variables numéricas → Mediana (robusta a outliers)

Variables categóricas → Moda (valor más frecuente)

In [11]:
for col in df.columns:
    if df[col].isnull().sum() > 0 and col != 'DEFAULT':
        
        if df[col].dtype in ['int64', 'float64']:
            median_val = df[col].median()
            df[col] = df[col].fillna(median_val)
            print(f"  {col}: mediana = {median_val:.2f}")
            
        else:
            mode_val = df[col].mode()
            if len(mode_val) > 0:
                df[col] = df[col].fillna(mode_val[0])
                print(f"  {col}: moda = '{mode_val[0]}'")
            else:
                df[col] = df[col].fillna('Unknown')
                print(f"  {col}: Unknown")

print(f"\n✅ Nulos restantes: {df.isnull().sum().sum()}")

  VerificationType: mediana = 4.00
  Gender: mediana = 0.00
  MonthlyPayment: mediana = 101.13
  City: moda = 'Tallinn'
  Education: mediana = 4.00
  MaritalStatus: mediana = -1.00
  EmploymentStatus: mediana = -1.00
  EmploymentDurationCurrentEmployer: moda = 'MoreThan5Years'
  OccupationArea: mediana = -1.00
  HomeOwnershipType: mediana = 3.00
  DebtToIncome: mediana = 0.00
  FreeCash: mediana = 0.00
  ModelVersion: mediana = 5.00
  Rating: moda = 'F'

✅ Nulos restantes: 0


In [12]:
null_counts = df.isnull().sum()
null_counts = null_counts[null_counts > 0].sort_values(ascending=False)

print(f"Columnas con nulos restantes: {len(null_counts)}")
print(null_counts.head(10))

Columnas con nulos restantes: 0
Series([], dtype: int64)


## 6. Ingeniería de Features


### 6.1 Verificación de Features Existentes

In [13]:
# Algunas Metricas Adicionales

# Relación Deuda/Ingreso (ya existe DebtToIncome, pero verificamos)
if 'DebtToIncome' in df.columns:
    print(f"DebtToIncome existe, rango: [{df['DebtToIncome'].min():.2f}, {df['DebtToIncome'].max():.2f}]")

# Tasa de interés relativa al monto
if 'Interest' in df.columns and 'Amount' in df.columns:
    df['Interest_per_Amount'] = df['Interest'] / df['Amount']
    print("Feature creada: Interest_per_Amount")

# Edad al momento del préstamo (ya existe Age)
if 'Age' in df.columns:
    print(f"Age existe, rango: [{df['Age'].min()}, {df['Age'].max()}] años")

DebtToIncome existe, rango: [0.00, 198.02]
Feature creada: Interest_per_Amount
Age existe, rango: [0, 77] años


## 7. Tratamiento de Outliers

### 7.1 Función de Tratamiento de Outliers

In [14]:


def tratar_outliers(df, columna, limite_inferior=None, limite_superior=None, metodo='capping'):
    """
    Trata outliers en una columna
    metodo: 'capping' (limitar), 'eliminar', 'mediana'
    """
    if columna not in df.columns:
        return df
    
    # Valores originales
    q1 = df[columna].quantile(0.01)  # Percentil 1
    q99 = df[columna].quantile(0.99) # Percentil 99
    
    if limite_inferior is None:
        limite_inferior = q1
    if limite_superior is None:
        limite_superior = q99
    
    outliers = ((df[columna] < limite_inferior) | (df[columna] > limite_superior)).sum()
    print(f"  {columna}: {outliers} outliers detectados")
    
    if metodo == 'capping':
        # Limitar valores extremos a los percentiles
        df[columna] = df[columna].clip(limite_inferior, limite_superior)
    elif metodo == 'mediana':
        df.loc[df[columna] < limite_inferior, columna] = df[columna].median()
        df.loc[df[columna] > limite_superior, columna] = df[columna].median()
    elif metodo == 'eliminar':
        df = df[(df[columna] >= limite_inferior) & (df[columna] <= limite_superior)]
    
    return df

print("="*50)
print("TRATAMIENTO DE OUTLIERS")
print("="*50)

# Tratar DebtToIncome (valores >10 no tienen sentido)
df = tratar_outliers(df, 'DebtToIncome', limite_inferior=0, limite_superior=10, metodo='capping')

# Tratar Age (debe estar entre 18 y 100)
df = tratar_outliers(df, 'Age', limite_inferior=18, limite_superior=100, metodo='capping')

# Tratar LoanDuration (meses, normalmente entre 1 y 60)
if 'LoanDuration' in df.columns:
    df = tratar_outliers(df, 'LoanDuration', limite_inferior=1, limite_superior=60, metodo='capping')

# Tratar Interest (tasas de interés, normalmente entre 0% y 50%)
if 'Interest' in df.columns:
    df = tratar_outliers(df, 'Interest', limite_inferior=0, limite_superior=50, metodo='capping')

# Tratar Amount (montos de préstamo)
if 'Amount' in df.columns:
    q1 = df['Amount'].quantile(0.01)
    q99 = df['Amount'].quantile(0.99)
    df = tratar_outliers(df, 'Amount', limite_inferior=q1, limite_superior=q99, metodo='capping')

print("\n✅ Outliers tratados")

TRATAMIENTO DE OUTLIERS
  DebtToIncome: 26244 outliers detectados
  Age: 53 outliers detectados
  LoanDuration: 0 outliers detectados
  Interest: 19479 outliers detectados
  Amount: 794 outliers detectados

✅ Outliers tratados


## 8. Verificación del Balanceo de Clases

In [15]:
# Balanceo de la Información

default_counts = df['DEFAULT'].value_counts()
print(f"Clase 0 (No default): {default_counts[0]:,} ({default_counts[0]/len(df)*100:.1f}%)")
print(f"Clase 1 (Default): {default_counts[1]:,} ({default_counts[1]/len(df)*100:.1f}%)")

if default_counts[1] / default_counts[0] < 0.2 or default_counts[0] / default_counts[1] < 0.2:
    print("⚠️ Dataset desbalanceado. Se necesitará técnica especial (SMOTE, class_weight)")
else:
    print("✅ Dataset relativamente balanceado")


Clase 0 (No default): 34,600 (44.7%)
Clase 1 (Default): 42,794 (55.3%)
✅ Dataset relativamente balanceado


## 9. Resumen Final del Preprocesamiento

In [16]:
print("="*50)
print("RESUMEN FINAL DEL PREPROCESAMIENTO")
print("="*50)
print(f"Filas: {df.shape[0]:,}")
print(f"Columnas: {df.shape[1]}")
print(f"Tasa de default: {df['DEFAULT'].mean()*100:.1f}%")
print(f"Columnas: {df.columns.tolist()}")

RESUMEN FINAL DEL PREPROCESAMIENTO
Filas: 77,394
Columnas: 54
Tasa de default: 55.3%
Columnas: ['BidsPortfolioManager', 'BidsApi', 'BidsManual', 'NewCreditCustomer', 'ApplicationSignedHour', 'ApplicationSignedWeekday', 'VerificationType', 'LanguageCode', 'Age', 'DateOfBirth', 'Gender', 'Country', 'AppliedAmount', 'Amount', 'Interest', 'LoanDuration', 'MonthlyPayment', 'City', 'UseOfLoan', 'Education', 'MaritalStatus', 'EmploymentStatus', 'EmploymentDurationCurrentEmployer', 'OccupationArea', 'HomeOwnershipType', 'IncomeFromPrincipalEmployer', 'IncomeFromPension', 'IncomeFromFamilyAllowance', 'IncomeFromSocialWelfare', 'IncomeFromLeavePay', 'IncomeFromChildSupport', 'IncomeOther', 'IncomeTotal', 'ExistingLiabilities', 'LiabilitiesTotal', 'RefinanceLiabilities', 'DebtToIncome', 'FreeCash', 'MonthlyPaymentDay', 'ActiveScheduleFirstPaymentReached', 'PlannedInterestTillDate', 'ModelVersion', 'Rating', 'Status', 'Restructured', 'PrincipalPaymentsMade', 'InterestAndPenaltyPaymentsMade', 'Prin

In [17]:
# Guardar en CSV
df.to_csv('../data/processed/bondora_processed.csv', index=False)
print("✅ Dataset guardado en data/processed/bondora_processed.csv")

✅ Dataset guardado en data/processed/bondora_processed.csv
